# Tutorial 2 – JSON exercise
**Intro to Cloud Computing – Braude**
Muhammad Khamra & Hamdan Khatib

What this notebook does:
1. Mounts Google Drive.
2. Creates `students.json` on Drive – every student in the group is one JSON object.
3. Shows a form: pick a student from the dropdown and their details appear in text boxes.
4. Lets you type a favorite program for the selected student and save it back to the JSON file with the **Update** button.

Run the cells from top to bottom.

## Step 1 – Mount Google Drive
Same as in the tutorial: the file will live in the root of *My Drive*.

In [ ]:
import json
import os
from google.colab import drive

drive.mount('/content/drive')

FILE_PATH = '/content/drive/MyDrive/students.json'

## Step 2 – Create `students.json`
Each student is a JSON object (name, last name, Braude email, list of courses, interesting link).
`favorite_program` starts empty and is filled in from the form.

The file is only written if it doesn't exist yet, so running this cell again won't erase favorite programs you already saved.
Set `RESET = True` to start over from the data below.

In [ ]:
students_data = {
    "students": [
        {
            "first_name": "Muhammad",
            "last_name": "Khamra",
            "email": "Muhammad.khamra@e.braude.ac.il",
            "courses": ["מחשוב ענן", "בינה מלאכותית", "WEB"],
            "interesting_link": "https://w3.braude.ac.il/#",
            "favorite_program": ""
        },
        {
            "first_name": "Hamdan",
            "last_name": "Khatib",
            "email": "Hamdan.Khatib@e.braude.ac.il",
            "courses": ["מחשוב ענן", "בינה מלאכותית", "WEB"],
            "interesting_link": "https://w3.braude.ac.il/#",
            "favorite_program": ""
        }
    ]
}

RESET = False

if RESET or not os.path.exists(FILE_PATH):
    with open(FILE_PATH, 'w', encoding='utf-8') as fid:
        json.dump(students_data, fid, indent=4, ensure_ascii=False)   # serialization (dump)
    print('Created', FILE_PATH)
else:
    print(FILE_PATH, 'already exists – keeping it as is')

## Step 3 – Helper functions
`load_students()` reads the file (deserialization with `json.load`), `save_students()` writes it back (serialization with `json.dump`).
`ensure_ascii=False` keeps the Hebrew course names readable inside the file.

In [ ]:
def load_students():
    with open(FILE_PATH, 'r', encoding='utf-8') as fid:
        return json.load(fid)


def save_students(data):
    with open(FILE_PATH, 'w', encoding='utf-8') as fid:
        json.dump(data, fid, indent=4, ensure_ascii=False)


def find_student(data, email):
    # The email is unique, so we use it to find the student's record
    for student in data['students']:
        if student['email'] == email:
            return student
    return None


def full_name(student):
    return student['first_name'] + ' ' + student['last_name']

## Step 4 – The form
* **Dropdown** – lists the students from the JSON file.
* Picking a student fills the **text boxes** with their details (read straight from the file).
* **Favorite program** box + **Update** button – saves the value into that student's record in `students.json` using `dict.update()`, like in the tutorial.

In [ ]:
import html
import ipywidgets as widgets
from IPython.display import display, HTML

MAIN_COLOR = '#1f4e79'

# A bit of styling for the form
display(HTML(f"""
<style>
  .student-form {{ background: #f4f8fc; border-radius: 12px; }}
  .student-form .widget-label {{ font-weight: 600; color: {MAIN_COLOR}; }}
</style>
"""))

students = load_students()['students']

label_style = {'description_width': '140px'}
field_layout = widgets.Layout(width='500px')

title = widgets.HTML(
    f"<div style='background:{MAIN_COLOR}; color:white; padding:14px 18px; border-radius:10px;"
    f" font-size:20px; font-weight:bold;'>🎓 Students – Cloud Computing, Tutorial 2</div>"
)

student_picker = widgets.Dropdown(
    options=[('-- choose a student --', None)] + [(full_name(s), s['email']) for s in students],
    value=None,
    description='Student:',
    style=label_style,
    layout=field_layout,
)

first_name_box = widgets.Text(description='First name:', disabled=True, style=label_style, layout=field_layout)
last_name_box = widgets.Text(description='Last name:', disabled=True, style=label_style, layout=field_layout)
email_box = widgets.Text(description='Braude email:', disabled=True, style=label_style, layout=field_layout)
courses_box = widgets.Textarea(description='Courses:', disabled=True, style=label_style,
                               layout=widgets.Layout(width='500px', height='85px'))
link_box = widgets.Text(description='Interesting link:', disabled=True, style=label_style, layout=field_layout)
link_open = widgets.HTML()

favorite_box = widgets.Text(description='Favorite program:', placeholder='Type a favorite program...',
                            disabled=True, style=label_style, layout=field_layout)
update_button = widgets.Button(description='Update', button_style='success', icon='save',
                               disabled=True, layout=widgets.Layout(width='150px', margin='6px 0 0 144px'))
status = widgets.HTML()

detail_boxes = [first_name_box, last_name_box, email_box, courses_box, link_box]


def show_student(change):
    email = student_picker.value
    status.value = ''

    if email is None:   # nothing selected – clear the form
        for box in detail_boxes:
            box.value = ''
        favorite_box.value = ''
        link_open.value = ''
        favorite_box.disabled = True
        update_button.disabled = True
        return

    student = find_student(load_students(), email)   # always read the latest data from the file
    first_name_box.value = student['first_name']
    last_name_box.value = student['last_name']
    email_box.value = student['email']
    courses_box.value = '\n'.join(student['courses'])
    link_box.value = student['interesting_link']
    link_open.value = (f"<div style='margin-left:144px'><a href='{html.escape(student['interesting_link'])}'"
                       f" target='_blank'>Open link ↗</a></div>")
    favorite_box.value = student.get('favorite_program', '')
    favorite_box.disabled = False
    update_button.disabled = False


def update_favorite(button):
    email = student_picker.value
    if email is None:
        return

    new_program = favorite_box.value.strip()
    if not new_program:
        status.value = "<div style='margin-left:144px; color:#c0392b'>Please type a program first.</div>"
        return

    data = load_students()
    student = find_student(data, email)
    student.update({'favorite_program': new_program})   # update the record
    save_students(data)                                  # write it back to students.json

    status.value = (f"<div style='margin-left:144px; color:#1e8449'>✔ Saved! {html.escape(full_name(student))}'s"
                    f" favorite program is now <b>{html.escape(new_program)}</b>.</div>")


student_picker.observe(show_student, names='value')
update_button.on_click(update_favorite)

form = widgets.VBox(
    [title, student_picker, widgets.HTML('<hr>'),
     first_name_box, last_name_box, email_box, courses_box, link_box, link_open,
     widgets.HTML('<hr>'),
     favorite_box, update_button, status],
    layout=widgets.Layout(border=f'2px solid {MAIN_COLOR}', padding='16px', width='580px'),
)
form.add_class('student-form')

display(form)

## Step 5 – Check the file
Run this after pressing **Update** to see that the favorite program was written into `students.json`.

In [ ]:
print(json.dumps(load_students(), indent=4, ensure_ascii=False))